In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import scipy.io

# ============================================================
# User settings
# ============================================================

# Reference / exact data candidates
REF_CANDIDATES = [
    "data/allen_cahn_python.mat",
    "data/allen_cahn.mat",
    "data/allen_cahn_python.npz",
    "data/allen_cahn_data.npz",

]

# IMPORTANT:
# You said these two names are reversed in meaning:
STO_SQP_PATH = "allen_cahn_stosqp_final_deter.npy"   # actually Sto-SQP
DET_SQP_PATH = "allen_cahn_stosqp_final.npy"         # actually deterministic SQP

ALM_PATH = "traditional_alm_allen_cahn_best.npy"

# Set this to your Adam saved model file
# Examples you may need to try:
#   "ac_adam_model.npy"
#   "allen_cahn_adam_model.npy"
#   "model_ac_adam.npy"
#   "hist_ac_adam_best.npy"  <-- only if this truly stores model params
ADAM_MODEL_PATH = "ac_adam_model.npy"

# Architecture assumptions for SQP / ALM / possibly Adam-flat-theta
HIDDEN_DIM = 30
NUM_HIDDEN = 3
LAYER_SIZES = [2] + [HIDDEN_DIM] * NUM_HIDDEN + [1]

# Activation
ACTIVATION = "tanh"

# Save names
OUT_PNG = "allen_cahn_heatmaps_predictions_and_errors.png"
OUT_PDF = "allen_cahn_heatmaps_predictions_and_errors.pdf"

# Figure titles
METHOD_NAMES = ["Sto-SQP", "Deterministic SQP", "ALM", "Adam"]

# ============================================================
# Utilities
# ============================================================

def find_existing_file(candidates):
    for c in candidates:
        if Path(c).exists():
            return c
    return None


def rel_l2(pred, true):
    pred = np.asarray(pred, dtype=np.float64)
    true = np.asarray(true, dtype=np.float64)
    return np.sqrt(np.mean((pred - true) ** 2) / (np.mean(true ** 2) + 1e-30))


def mse(pred, true):
    pred = np.asarray(pred, dtype=np.float64)
    true = np.asarray(true, dtype=np.float64)
    return np.mean((pred - true) ** 2)


# ============================================================
# Load reference / exact data
# ============================================================

def load_reference():
    ref_path = find_existing_file(REF_CANDIDATES)
    if ref_path is None:
        raise FileNotFoundError(
            "Could not find reference file. Tried:\n" + "\n".join(REF_CANDIDATES)
        )

    print(f"Loading reference data from: {ref_path}")

    if ref_path.endswith(".mat"):
        data = scipy.io.loadmat(ref_path)
        t = np.asarray(data["t"]).squeeze().astype(np.float64)
        x = np.asarray(data["x"]).squeeze().astype(np.float64)
        usol = np.asarray(data["usol"]).astype(np.float64)

    elif ref_path.endswith(".npz"):
        data = np.load(ref_path, allow_pickle=True)
        if "t" in data:
            t = np.asarray(data["t"], dtype=np.float64).squeeze()
        elif "t_star" in data:
            t = np.asarray(data["t_star"], dtype=np.float64).squeeze()
        else:
            raise KeyError("Could not find time array in npz")

        if "x" in data:
            x = np.asarray(data["x"], dtype=np.float64).squeeze()
        elif "x_star" in data:
            x = np.asarray(data["x_star"], dtype=np.float64).squeeze()
        else:
            raise KeyError("Could not find space array in npz")

        if "usol" in data:
            usol = np.asarray(data["usol"], dtype=np.float64)
        elif "u_ref" in data:
            usol = np.asarray(data["u_ref"], dtype=np.float64)
        else:
            raise KeyError("Could not find solution array in npz")
    else:
        raise ValueError(f"Unsupported reference file type: {ref_path}")

    # Ensure shape is (nt, nx)
    if usol.shape[0] != len(t) and usol.shape[1] == len(t):
        usol = usol.T

    if usol.shape != (len(t), len(x)):
        raise ValueError(
            f"Reference solution shape mismatch: usol.shape={usol.shape}, "
            f"expected {(len(t), len(x))}"
        )

    return x, t, usol


# ============================================================
# MLP helpers for flat-theta models (SQP / ALM)
# ============================================================

def normalize_xt(X, x_min, x_max, t_min, t_max):
    xx = X[:, 0:1]
    tt = X[:, 1:2]

    x_n = 2.0 * (xx - x_min) / (x_max - x_min + 1e-30) - 1.0
    t_n = 2.0 * (tt - t_min) / (t_max - t_min + 1e-30) - 1.0
    return np.concatenate([x_n, t_n], axis=1)


def unflatten_theta(theta, layer_sizes):
    params = []
    idx = 0
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        w_size = m * n
        b_size = n

        W = theta[idx:idx + w_size].reshape((m, n))
        idx += w_size
        b = theta[idx:idx + b_size].reshape((n,))
        idx += b_size

        params.append((W, b))

    if idx != theta.size:
        raise ValueError(f"Did not consume all theta entries: used {idx}, total {theta.size}")

    return params


def activate(z, name="tanh"):
    if name == "tanh":
        return np.tanh(z)
    elif name == "silu":
        return z / (1.0 + np.exp(-z))
    else:
        raise ValueError(f"Unsupported activation: {name}")


def mlp_apply_param_list(params, X, activation="tanh"):
    h = X
    for i, (W, b) in enumerate(params):
        h = h @ W + b
        if i < len(params) - 1:
            h = activate(h, activation)
    return h


def predict_from_flat_theta(theta, layer_sizes, x, t, activation="tanh"):
    x_min, x_max = float(np.min(x)), float(np.max(x))
    t_min, t_max = float(np.min(t)), float(np.max(t))

    XX, TT = np.meshgrid(x, t, indexing="xy")
    X_eval = np.stack([XX.reshape(-1), TT.reshape(-1)], axis=1)
    Xn = normalize_xt(X_eval, x_min, x_max, t_min, t_max)

    params = unflatten_theta(theta, layer_sizes)
    pred = mlp_apply_param_list(params, Xn, activation=activation).reshape(len(t), len(x))
    return pred


# ============================================================
# Generic Adam loader
# Supports several common formats:
# 1) flat 1D theta array
# 2) npy object array containing dict/list
# 3) dict with key "params"
# 4) param list [(W,b), ...]
# ============================================================

def convert_to_param_list(obj):
    # unwrap 0-d object arrays
    if isinstance(obj, np.ndarray) and obj.dtype == object and obj.shape == ():
        obj = obj.item()

    # dict cases
    if isinstance(obj, dict):
        if "params" in obj:
            return convert_to_param_list(obj["params"])

        # common custom dict format: {"weights":[...], "biases":[...]}
        if "weights" in obj and "biases" in obj:
            Ws = obj["weights"]
            bs = obj["biases"]
            return [(np.asarray(W, dtype=np.float64), np.asarray(b, dtype=np.float64))
                    for W, b in zip(Ws, bs)]

        # maybe layers stored directly
        if "layers" in obj:
            return convert_to_param_list(obj["layers"])

    # list/tuple of layers
    if isinstance(obj, (list, tuple)):
        if len(obj) == 0:
            raise ValueError("Empty parameter list")

        # already [(W,b), ...]
        first = obj[0]
        if isinstance(first, (list, tuple)) and len(first) == 2:
            return [(np.asarray(W, dtype=np.float64), np.asarray(b, dtype=np.float64))
                    for W, b in obj]

        # maybe [{"W":..., "b":...}, ...]
        if isinstance(first, dict) and "W" in first and "b" in first:
            return [(np.asarray(layer["W"], dtype=np.float64),
                     np.asarray(layer["b"], dtype=np.float64))
                    for layer in obj]

    raise ValueError("Could not convert object to parameter list")


def predict_from_adam_model(path, x, t, activation="tanh"):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Adam model file not found: {path}\n"
            f"Please set ADAM_MODEL_PATH correctly."
        )

    obj = np.load(path, allow_pickle=True)

    # Case 1: plain 1D theta
    if isinstance(obj, np.ndarray) and obj.dtype != object and obj.ndim == 1:
        print("Adam model interpreted as flat theta vector.")
        return predict_from_flat_theta(obj.astype(np.float64), LAYER_SIZES, x, t, activation=activation)

    # Case 2: maybe npz-like or object dict/list
    try:
        params = convert_to_param_list(obj)
        print("Adam model interpreted as parameter list / dict format.")
    except Exception:
        # if np.load returned NpzFile
        if hasattr(obj, "files"):
            candidate_dict = {k: obj[k] for k in obj.files}
            params = convert_to_param_list(candidate_dict)
            print("Adam model interpreted as NPZ dict format.")
        else:
            raise

    x_min, x_max = float(np.min(x)), float(np.max(x))
    t_min, t_max = float(np.min(t)), float(np.max(t))

    XX, TT = np.meshgrid(x, t, indexing="xy")
    X_eval = np.stack([XX.reshape(-1), TT.reshape(-1)], axis=1)
    Xn = normalize_xt(X_eval, x_min, x_max, t_min, t_max)

    pred = mlp_apply_param_list(params, Xn, activation=activation).reshape(len(t), len(x))
    return pred


# ============================================================
# Load all models
# ============================================================

def load_and_predict_all(x, t):
    preds = {}

    # Sto-SQP
    theta_sto = np.asarray(np.load(STO_SQP_PATH), dtype=np.float64).reshape(-1)
    preds["Sto-SQP"] = predict_from_flat_theta(theta_sto, LAYER_SIZES, x, t, activation=ACTIVATION)

    # Deterministic SQP
    theta_det = np.asarray(np.load(DET_SQP_PATH), dtype=np.float64).reshape(-1)
    preds["Deterministic SQP"] = predict_from_flat_theta(theta_det, LAYER_SIZES, x, t, activation=ACTIVATION)

    # ALM
    theta_alm = np.asarray(np.load(ALM_PATH), dtype=np.float64).reshape(-1)
    preds["ALM"] = predict_from_flat_theta(theta_alm, LAYER_SIZES, x, t, activation=ACTIVATION)

    # Adam
    preds["Adam"] = predict_from_adam_model(ADAM_MODEL_PATH, x, t, activation=ACTIVATION)

    return preds


# ============================================================
# Main
# ============================================================

x, t, u_ref = load_reference()
preds = load_and_predict_all(x, t)

# Compute metrics
metrics = {}
errors = {}

for name, u_pred in preds.items():
    errors[name] = np.abs(u_pred - u_ref)
    metrics[name] = {
        "mse": mse(u_pred, u_ref),
        "relL2": rel_l2(u_pred, u_ref),
    }

print("\n===== Metrics =====")
for name in METHOD_NAMES:
    print(
        f"{name:18s}  MSE = {metrics[name]['mse']:.6e}   relL2 = {metrics[name]['relL2']:.6e}"
    )

# Shared color scales
all_sol = [u_ref] + [preds[name] for name in METHOD_NAMES]
sol_vmin = min(np.min(a) for a in all_sol)
sol_vmax = max(np.max(a) for a in all_sol)

all_err = [errors[name] for name in METHOD_NAMES]
err_vmin = 0.0
err_vmax = max(np.max(a) for a in all_err)

# ============================================================
# Plot
# ============================================================

fig, axes = plt.subplots(2, 5, figsize=(24, 9), constrained_layout=True)

extent = [float(np.min(x)), float(np.max(x)), float(np.min(t)), float(np.max(t))]

# ----------------------------
# Top row: exact + predictions
# ----------------------------
im0 = axes[0, 0].imshow(
    u_ref,
    origin="lower",
    aspect="auto",
    extent=extent,
    vmin=sol_vmin,
    vmax=sol_vmax,
    cmap="viridis",
)
axes[0, 0].set_title("Exact / Reference")
axes[0, 0].set_xlabel("x")
axes[0, 0].set_ylabel("t")

for j, name in enumerate(METHOD_NAMES, start=1):
    im = axes[0, j].imshow(
        preds[name],
        origin="lower",
        aspect="auto",
        extent=extent,
        vmin=sol_vmin,
        vmax=sol_vmax,
        cmap="viridis",
    )
    axes[0, j].set_title(
        f"{name}\nrelL2={metrics[name]['relL2']:.3e}"
    )
    axes[0, j].set_xlabel("x")
    axes[0, j].set_ylabel("t")

# ----------------------------
# Bottom row: label panel + errors
# ----------------------------
axes[1, 0].axis("off")
axes[1, 0].text(
    0.5, 0.5,
    "Absolute error heatmaps\n$|u_{pred} - u_{exact}|$",
    ha="center", va="center", fontsize=14
)

for j, name in enumerate(METHOD_NAMES, start=1):
    im_err = axes[1, j].imshow(
        errors[name],
        origin="lower",
        aspect="auto",
        extent=extent,
        vmin=err_vmin,
        vmax=err_vmax,
        cmap="magma",
    )
    axes[1, j].set_title(
        f"{name} error\nMSE={metrics[name]['mse']:.3e}"
    )
    axes[1, j].set_xlabel("x")
    axes[1, j].set_ylabel("t")

# Shared colorbars
cbar1 = fig.colorbar(im0, ax=axes[0, :], shrink=0.90, pad=0.02)
cbar1.set_label("u(t, x)")

cbar2 = fig.colorbar(im_err, ax=axes[1, 1:], shrink=0.90, pad=0.02)
cbar2.set_label(r"$|u_{pred} - u_{exact}|$")

fig.suptitle("Allen-Cahn: exact solution, predictions, and absolute errors", fontsize=16)

fig.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
fig.savefig(OUT_PDF, bbox_inches="tight")
print(f"\nSaved figure -> {OUT_PNG}")
print(f"Saved figure -> {OUT_PDF}")

plt.show()

FileNotFoundError: Could not find reference file. Tried:
allen_cahn_python.mat
allen_cahn.mat
allen_cahn_python.npz
allen_cahn_data.npz

Found 5 .npz files.
[SKIP] ./1/advection_c40_data.npz
"Could not identify keys in ./1/advection_c40_data.npz. Available keys: ['u_ref', 't_star', 'x_star', 'c', 'T', 'L']"
[SKIP] ./1/advection_c60_data.npz
"Could not identify keys in ./1/advection_c60_data.npz. Available keys: ['u_ref', 't_star', 'x_star', 'c', 'T', 'L']"
[SKIP] ./1/advection_wave_data.npz
"Could not identify keys in ./1/advection_wave_data.npz. Available keys: ['u_ref', 't_star', 'x_star', 'c', 'T', 'L']"
[SKIP] ./moose/reference_data/no_adv_320x256/single_T_no_adv_320x256_reference.npz
"Could not identify keys in ./moose/reference_data/no_adv_320x256/single_T_no_adv_320x256_reference.npz. Available keys: ['x', 'y', 'time', 'T', 'phi', 'fuel_nodes', 'coolant_nodes', 'interface_nodes', 'params']"
[SKIP] ./moose/sqp_dataset.npz
"Could not identify keys in ./moose/sqp_dataset.npz. Available keys: ['t', 'x', 'y', 'phi', 'T', 'phi_fuel', 'Ts', 'Tf', 'T_interface']"
Saved summary to heatmaps/summary.csv
